# Airbnb Pricing & Market Intelligence

## Data Audit

This notebook evaluates the structure and quality of the datasets before data cleaning, modeling, and analysis.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

listings = pd.read_csv(
    "../data/raw/Listings.csv",
    encoding="latin1"
)

reviews = pd.read_csv(
    "../data/raw/Reviews.csv",
    encoding="latin1"
)

print("Datasets loaded successfully.")

C:\Users\Hp\AppData\Local\Temp\ipykernel_12824\438250735.py:6: DtypeWarning: Columns (0: host_response_time, 1: district) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv(


Datasets loaded successfully.


## Structural Audit

The objective of this section is to understand the structure, size, and granularity of the datasets before evaluating data quality.

In [2]:
# Listings Dataset Structure

print("LISTINGS DATASET")
print("-" * 50)

print(f"Rows    : {listings.shape[0]:,}")
print(f"Columns : {listings.shape[1]:,}")

print("\nColumn Names:\n")
print(list(listings.columns))


print("\n" + "=" * 70 + "\n")


# Reviews Dataset Structure

print("REVIEWS DATASET")
print("-" * 50)

print(f"Rows    : {reviews.shape[0]:,}")
print(f"Columns : {reviews.shape[1]:,}")

print("\nColumn Names:\n")
print(list(reviews.columns))

LISTINGS DATASET
--------------------------------------------------
Rows    : 279,712
Columns : 33

Column Names:

['listing_id', 'name', 'host_id', 'host_since', 'host_location', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_total_listings_count', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'district', 'city', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bedrooms', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable']


REVIEWS DATASET
--------------------------------------------------
Rows    : 5,373,143
Columns : 4

Column Names:

['listing_id', 'review_id', 'date', 'reviewer_id']


### Dataset Granularity and Key Assessment

This section identifies the level of detail represented by each dataset and highlights potential key columns that may be used for uniqueness and relationships.

In [3]:
# Potential Keys and Dataset Grain

print("LISTINGS")
print("-" * 50)

print("Unique listing_id :", listings["listing_id"].nunique())
print("Total rows        :", len(listings))

print("\nREVIEWS")
print("-" * 50)

print("Unique review_id  :", reviews["review_id"].nunique())
print("Total rows        :", len(reviews))

print("\nUnique listing_id in Reviews :", reviews["listing_id"].nunique())

LISTINGS
--------------------------------------------------
Unique listing_id : 279712
Total rows        : 279712

REVIEWS
--------------------------------------------------
Unique review_id  : 5372983
Total rows        : 5373143

Unique listing_id in Reviews : 193556


### Key Integrity Validation

This section verifies whether candidate key columns contain duplicate values that could affect data quality and downstream analysis.

In [4]:
# Duplicate check for candidate keys

print("Duplicate listing_id in Listings:")
print(listings["listing_id"].duplicated().sum())

print("\nDuplicate review_id in Reviews:")
print(reviews["review_id"].duplicated().sum())

Duplicate listing_id in Listings:
0

Duplicate review_id in Reviews:
160


## Missing Value Audit

This section evaluates the completeness of each dataset by identifying missing values and measuring their impact across columns.

In [5]:
# Missing Values - Listings

listings_missing = (
    listings.isna()
            .sum()
            .reset_index()
)

listings_missing.columns = ["column", "missing_count"]

listings_missing["missing_percent"] = (
    listings_missing["missing_count"] / len(listings) * 100
).round(2)

listings_missing = listings_missing.sort_values(
    by="missing_count",
    ascending=False
)

listings_missing[listings_missing["missing_count"] > 0]

,column,missing_count,missing_percent
13,district,242700,86.77
5,host_response_time,128782,46.04
6,host_response_rate,128782,46.04
7,host_acceptance_rate,113087,40.43
31,review_scores_value,91785,32.81
30,review_scores_location,91775,32.81
28,review_scores_checkin,91771,32.81
26,review_scores_accuracy,91713,32.79
29,review_scores_communication,91687,32.78
27,review_scores_cleanliness,91665,32.77


In [6]:
# Missing Values - Reviews

reviews_missing = (
    reviews.isna()
           .sum()
           .reset_index()
)

reviews_missing.columns = ["column", "missing_count"]

reviews_missing["missing_percent"] = (
    reviews_missing["missing_count"] / len(reviews) * 100
).round(2)

reviews_missing.sort_values(
    by="missing_count",
    ascending=False
)

,column,missing_count,missing_percent
0,listing_id,0,0.0
1,review_id,0,0.0
2,date,0,0.0
3,reviewer_id,0,0.0


## Duplicate Audit

This section identifies duplicate records that may impact data quality, aggregation accuracy, and relationship integrity.

In [7]:
# Full Row Duplicate Check

print("LISTINGS")
print("-" * 50)
print("Duplicate Rows:", listings.duplicated().sum())

print("\nREVIEWS")
print("-" * 50)
print("Duplicate Rows:", reviews.duplicated().sum())

LISTINGS
--------------------------------------------------
Duplicate Rows: 0

REVIEWS
--------------------------------------------------
Duplicate Rows: 0


In [8]:
# Duplicate review_id Investigation

duplicate_review_ids = (
    reviews["review_id"]
    .value_counts()
    .loc[lambda x: x > 1]
)

print("Number of duplicate review_ids:")
print(len(duplicate_review_ids))

print("\nTop duplicate review_ids:")
print(duplicate_review_ids.head(10))

Number of duplicate review_ids:
160

Top duplicate review_ids:
review_id
529009867    2
470333123    2
545322016    2
483309396    2
474335317    2
455861702    2
455738773    2
455177386    2
447868466    2
451736404    2
Name: count, dtype: int64


In [9]:
# Inspect Duplicate review_id Records

duplicate_ids = reviews.loc[
    reviews["review_id"].duplicated(keep=False),
    "review_id"
]

reviews.loc[
    reviews["review_id"].isin(duplicate_ids)
].sort_values("review_id").head(20)

,listing_id,review_id,date,reviewer_id
4549014,22219088,200088015,2017-10-04,123346683
4548449,2997346,200088015,2017-10-04,123346683
4431427,22219088,205182329,2017-10-21,23770481
4430927,2997346,205182329,2017-10-21,23770481
4284366,22219088,223954194,2018-01-02,77129337
4283655,2997346,223954194,2018-01-02,77129337
4564028,2997346,256197483,2018-04-22,115838328
4564678,22219088,256197483,2018-04-22,115838328
4403899,22219250,256940062,2018-04-24,172857558
4403226,466155,256940062,2018-04-24,172857558


## Data Type Audit

This section evaluates whether columns have appropriate data types and identifies potential type inconsistencies that may require cleaning.

In [10]:
# Data Types - Listings

listings.dtypes.to_frame("data_type")

,data_type
listing_id,int64
name,str
host_id,int64
host_since,str
host_location,str
host_response_time,str
host_response_rate,float64
host_acceptance_rate,float64
host_is_superhost,str
host_total_listings_count,float64


In [11]:
# Data Types - Reviews

reviews.dtypes.to_frame("data_type")

,data_type
listing_id,int64
review_id,int64
date,str
reviewer_id,int64


In [12]:
# Investigate Mixed-Type Warning Columns

print("host_response_time")
print("-" * 50)
print(listings["host_response_time"].dropna().unique()[:20])

print("\n")

print("district")
print("-" * 50)
print(listings["district"].dropna().unique()[:20])

host_response_time
--------------------------------------------------
<ArrowStringArray>
['within a few hours', 'within a day', 'within an hour', 'a few days or more']
Length: 4, dtype: str


district
--------------------------------------------------
<ArrowStringArray>
['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
Length: 5, dtype: str


## Domain Validity Audit

This section evaluates whether column values fall within expected business domains and identifies invalid or unexpected values.

In [13]:
# Categorical Domain Inspection

categorical_columns = [
    "host_response_time",
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "room_type",
    "instant_bookable"
]

for col in categorical_columns:
    print(f"\n{col}")
    print("-" * 50)
    print(listings[col].value_counts(dropna=False))


host_response_time
--------------------------------------------------
host_response_time
NaN                   128782
within an hour         83464
within a few hours     28891
within a day           23425
a few days or more     15150
Name: count, dtype: int64

host_is_superhost
--------------------------------------------------
host_is_superhost
f      229294
t       50253
NaN       165
Name: count, dtype: int64

host_has_profile_pic
--------------------------------------------------
host_has_profile_pic
t      278631
f         916
NaN       165
Name: count, dtype: int64

host_identity_verified
--------------------------------------------------
host_identity_verified
t      201191
f       78356
NaN       165
Name: count, dtype: int64

room_type
--------------------------------------------------
room_type
Entire place    182005
Private room     86988
Hotel room        5857
Shared room       4862
Name: count, dtype: int64

instant_bookable
-----------------------------------------------

In [14]:
# Domain Validation - Numeric Business Fields

numeric_columns = [
    "price",
    "accommodates",
    "bedrooms",
    "minimum_nights",
    "maximum_nights",
    "review_scores_rating"
]

listings[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
price,279712.0,608.792737,3.441827e+03,0.0,75.0,150.0,474.0,6.252160e+05
accommodates,279712.0,3.288736,2.133379e+00,0.0,2.0,2.0,4.0,1.600000e+01
bedrooms,250277.0,1.515509,1.153080e+00,1.0,1.0,1.0,2.0,5.000000e+01
minimum_nights,279712.0,8.050967,3.151895e+01,1.0,1.0,2.0,5.0,9.999000e+03
maximum_nights,279712.0,27558.596667,7.282875e+06,1.0,45.0,1125.0,1125.0,2.147484e+09
review_scores_rating,188307.0,93.405195,1.007044e+01,20.0,91.0,96.0,100.0,1.000000e+02


## Consistency Audit

This section evaluates consistency in categorical and geographic fields by identifying formatting issues, spelling variations, and duplicate categories caused by inconsistent data entry.

In [15]:
# Geographic Consistency Check

geo_columns = [
    "city",
    "district",
    "neighbourhood"
]

for col in geo_columns:
    print(f"\n{col}")
    print("-" * 50)
    print("Unique Values:", listings[col].nunique(dropna=True))


city
--------------------------------------------------
Unique Values: 10

district
--------------------------------------------------
Unique Values: 5

neighbourhood
--------------------------------------------------
Unique Values: 660


In [16]:
# Geographic Domain Inspection

for col in ["city", "district"]:
    print(f"\n{col}")
    print("-" * 50)
    print(sorted(listings[col].dropna().unique()))


city
--------------------------------------------------
['Bangkok', 'Cape Town', 'Hong Kong', 'Istanbul', 'Mexico City', 'New York', 'Paris', 'Rio de Janeiro', 'Rome', 'Sydney']

district
--------------------------------------------------
['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']


## Outlier Audit

This section identifies extreme values and potentially unrealistic observations that may affect statistical analysis and business insights.

In [17]:
# Numerical Summary for Outlier Detection

outlier_columns = [
    "price",
    "accommodates",
    "bedrooms",
    "minimum_nights",
    "maximum_nights"
]

listings[outlier_columns].describe(
    percentiles=[0.01, 0.05, 0.95, 0.99]
).T

,count,mean,std,min,1%,5%,95%,99%,max
price,279712.0,608.792737,3.441827e+03,0.0,25.0,40.0,2000.0,6500.0,6.252160e+05
accommodates,279712.0,3.288736,2.133379e+00,0.0,1.0,1.0,7.0,12.0,1.600000e+01
bedrooms,250277.0,1.515509,1.153080e+00,1.0,1.0,1.0,3.0,5.0,5.000000e+01
minimum_nights,279712.0,8.050967,3.151895e+01,1.0,1.0,1.0,30.0,80.0,9.999000e+03
maximum_nights,279712.0,27558.596667,7.282875e+06,1.0,4.0,10.0,1125.0,1125.0,2.147484e+09


In [18]:
# Potentially Invalid Values

print("Price <= 0")
print((listings["price"] <= 0).sum())

print("\nAccommodates <= 0")
print((listings["accommodates"] <= 0).sum())

print("\nBedrooms <= 0")
print((listings["bedrooms"] <= 0).sum())

print("\nMinimum Nights > 365")
print((listings["minimum_nights"] > 365).sum())

print("\nMaximum Nights > 3650")
print((listings["maximum_nights"] > 3650).sum())

Price <= 0
113

Accommodates <= 0
85

Bedrooms <= 0
0

Minimum Nights > 365
96

Maximum Nights > 3650
53


## Relationship Audit

This section validates the relationship between the Listings and Reviews datasets and identifies any referential integrity issues.

In [19]:
# Referential Integrity Check

listing_ids_in_listings = set(listings["listing_id"])
listing_ids_in_reviews = set(reviews["listing_id"])

orphan_reviews = listing_ids_in_reviews - listing_ids_in_listings

print("Listing IDs in Listings :", len(listing_ids_in_listings))
print("Listing IDs in Reviews  :", len(listing_ids_in_reviews))

print("\nOrphan Listing IDs in Reviews:")
print(len(orphan_reviews))

Listing IDs in Listings : 279712
Listing IDs in Reviews  : 193556

Orphan Listing IDs in Reviews:
0


## Final Audit Report

This section summarizes the key findings identified during the data audit process and highlights areas requiring attention during data cleaning and modeling.

In [20]:
audit_summary = pd.DataFrame({
    "Audit Area": [
        "Structural Audit",
        "Missing Value Audit",
        "Duplicate Audit",
        "Data Type Audit",
        "Domain Validity Audit",
        "Consistency Audit",
        "Outlier Audit",
        "Relationship Audit"
    ],
    "Key Finding": [
        "Unique listing_id in Listings. 160 duplicate review_ids found in Reviews.",
        "High missingness in district, host response metrics, and review score columns.",
        "No duplicate rows. Duplicate review_ids linked to multiple listing_ids.",
        "Date fields stored as text. Boolean attributes stored as categorical values.",
        "Zero-price listings, zero-accommodate listings, and extreme night limits detected.",
        "City and district values are consistently formatted.",
        "Extreme values detected in price, bedrooms, minimum_nights, and maximum_nights.",
        "No orphan review records. Referential integrity maintained."
    ]
})

print(audit_summary.to_string(index=False))

           Audit Area                                                                        Key Finding
     Structural Audit          Unique listing_id in Listings. 160 duplicate review_ids found in Reviews.
  Missing Value Audit     High missingness in district, host response metrics, and review score columns.
      Duplicate Audit            No duplicate rows. Duplicate review_ids linked to multiple listing_ids.
      Data Type Audit       Date fields stored as text. Boolean attributes stored as categorical values.
Domain Validity Audit Zero-price listings, zero-accommodate listings, and extreme night limits detected.
    Consistency Audit                               City and district values are consistently formatted.
        Outlier Audit    Extreme values detected in price, bedrooms, minimum_nights, and maximum_nights.
   Relationship Audit                        No orphan review records. Referential integrity maintained.


### Audit Conclusion

The audit identified several data quality issues requiring further investigation and treatment.

#### Key Findings

- High missingness exists in district, host response metrics, and review score columns.
- The Reviews dataset contained 160 duplicate review records linked to different listing_ids.
- Date fields are currently stored as text.
- Several boolean attributes are stored as categorical text values.
- 113 listings have a price of zero.
- 85 listings have accommodates equal to zero.
- Extreme values were observed in price, bedrooms, minimum_nights, and maximum_nights.
- Geographic fields are consistently formatted.
- Some review records reference listing_ids that do not exist in the Listings dataset, indicating referential integrity issues between Listings and Reviews.